# 08 — Historical and market-relative features: baseline classifiers

Compare original candle/calendar features against causal momentum, volatility, volume and leave-one-out universe-market features. **No random forests, boosted trees, regression, or neural networks.** Five expanding chronological folds; fold-specific target quantiles, K-means, encoders and scaling.

The market proxy is an equal-weight contemporaneous average of *other eligible stocks*, **not SPY**. The same test rows are used for both feature sets.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from stock_ml_dynamic.pooled_models import load_master
from stock_ml_dynamic.historical_market import run_feature_comparison, plot_feature_comparison
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'pyproject.toml').is_file()), None)
if ROOT is None: raise RuntimeError('Open notebook from inside stock_ml_dynamic repo')
INTERVAL = '1d'
required = ('master_labeled.parquet','latest_inference.parquet','transformers.joblib','manifest.json')
candidates = [ROOT/'artifacts'/'master_tables'/INTERVAL, ROOT/'notebooks'/'artifacts'/'master_tables'/INTERVAL]
MASTER = next((p for p in candidates if all((p/f).is_file() for f in required)), None)
if MASTER is None: raise FileNotFoundError(f'Run Notebook 03 to generate master artifacts. Checked: {candidates}')
master, latest, bundle, manifest = load_master(MASTER)
print('Master:', MASTER, '| rows:', len(master), '| symbols:', master.symbol.nunique())
print('Volume column available:', 'volume' in master)
display(master[['symbol','bar_timestamp','adj_close','next_return']].tail())

In [ ]:
N_FOLDS = 5
MAX_TRAIN_ROWS = 120_000  # training-only sample cap; validation uses all complete rows
OUTPUT = ROOT/'artifacts'/'historical_market_benchmark'/INTERVAL
scores, curves, eligibility, added_features = run_feature_comparison(
    master, n_folds=N_FOLDS, max_train_rows=MAX_TRAIN_ROWS, output_dir=OUTPUT)
print('New features:', len(added_features), added_features)
print('Excluded:', eligibility.loc[~eligibility.eligible, ['symbol','bars','latest_adj_z']].to_string(index=False))
display(scores)

In [ ]:
summary = plot_feature_comparison(scores, curves, OUTPUT)
display(summary.round(5))
paired = scores.pivot_table(index=['fold','model'], columns='feature_set', values=['accuracy','balanced_accuracy','roc_auc_ovr_macro'])
for metric in ('accuracy','balanced_accuracy','roc_auc_ovr_macro'):
    paired[(metric,'delta')] = paired[(metric,'historical_market')] - paired[(metric,'original')]
display(paired.round(5))
print('Saved artifacts:', OUTPUT)

## Interpretation and limits

- Accuracy and macro one-vs-rest ROC-AUC must be compared to **fold-specific** prior-class baseline, not a fixed 33% accuracy.
- Expanded features are computed using current or past bars only; no next-return labels enter features.
- Cross-sectional market proxy requires at least 10 other tickers with contemporaneous returns. Missing/partial sessions may reduce coverage.
- This experiment is exploratory: repeatedly selecting the best model using the same five folds can overfit model selection. Reserve a separate untouched recent test period before claiming generalization.
- Universe membership is defined by latest history eligibility, which can introduce survivorship bias in retrospective analysis.
- No external SPY or sector index is used; sector-relative signals require a separate correctly timestamped sector mapping and benchmark series.
- Full run trains **40 model fits** (5 folds × 2 feature sets × 4 classifiers), with fit/prediction timings logged.